In [8]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List, Annotated
from langchain_core.messages import HumanMessage, BaseMessage, AIMessage
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import MemorySaver

In [3]:
from langgraph.graph.message import add_messages
# create the state 
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage] , add_messages]

In [4]:
chat_model = ChatOllama(model = "phi4-mini:latest")

In [5]:
def chat_node(state: ChatState):
    global chat_model
    # take user query from state
    messages = state['messages']
    # send query to llm 
    response = chat_model.invoke(messages)
    # store the reponse
    return {
        "messages": [AIMessage(response.content)]
    }

In [9]:
checkpointer  = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node("chat_node" , chat_node)

graph.add_edge(START , "chat_node")
graph.add_edge("chat_node" , END)

chatbot = graph.compile(checkpointer = checkpointer)

In [12]:
thread_id = "1" # id for user

initial_state = {
    "messages": HumanMessage("What is the capital of bangladesh?")
}
config = {
    "configurable": {
        "thread_id": thread_id
    }
}
result = chatbot.invoke(initial_state , config = config)
result

{'messages': [HumanMessage(content='What is the capital of bangladesh?', additional_kwargs={}, response_metadata={}, id='55cfa97f-7e31-43b6-a128-94f2f57d389b'),
  AIMessage(content="The capital city of Bangladesh is Dhaka. It was also previously known as Dacca until 1982, when it reverted to its historical name after independence from Pakistan in 1971.\n\nDhaka serves not only as a political and administrative center but has grown into one of Asia's largest urban agglomerations with diverse industries including textiles, shipbuilding, steel mills, leather products manufacturing. The city is also known for the Dhaka University which was established by Sir Syed Ahmed Khan on August 25th in Dacca (now Dhaka).", additional_kwargs={}, response_metadata={}, id='096422cb-c574-4850-8e52-58e2b5e5a3bc', tool_calls=[], invalid_tool_calls=[])]}